
| Axis | **AI Agent** (singular) | **Agentic AI** (the system) |
|---|---|---|
| Scope | A component — one entity | An architecture / paradigm |
| "Minds" | Usually **one** LLM | **Multiple** coordinated, specialized agents |
| Control flow | A single goal-loop: *think → act → observe* | *Decompose → delegate → orchestrate → evaluate* |
| Memory | Task/session scratchpad | Shared/persistent across agents & steps |
| Recovery | Retry a tool, re-prompt itself | Critic/reflection, re-planning, handoffs |
| Demo | A coder that edits files & runs tests | A *team*: planner + coder + tester + reviewer |

In [1]:
# !pip install openai
import json, os
from pathlib import Path
from openai import OpenAI

import os, json
from dotenv import load_dotenv
import subprocess, time
from mcp.client.streamable_http import streamable_http_client
from contextlib import AsyncExitStack
from mcp import ClientSession


import textwrap

import truststore
truststore.inject_into_ssl()

def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv('/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/openai_key.env')  # reads .env file in the current directory

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")


client = OpenAI()
MODEL = "gpt-5-nano"   

API key loaded successfully.


# Part 1

In [2]:
# --- Tools the agent can choose to use ---
def web_search(query: str) -> str:
    # (mock) pretend we hit a real search engine
    return f"Top result for '{query}': Scaler is an edtech platform founded in 2019."

def calculator(expression: str) -> str:
    return str(eval(expression))   # demo only — never eval untrusted input in prod

TOOL_IMPL = {"web_search": web_search, "calculator": calculator}

In [3]:
# --- Tool schemas: how we describe the tools to the model ---
tools = [
    {"type": "function", "name": "web_search",
     "description": "Search the web and return the top result.",
     "parameters": {"type": "object",
                    "properties": {"query": {"type": "string"}},
                    "required": ["query"], "additionalProperties": False}},
    {"type": "function", "name": "calculator",
     "description": "Evaluate a basic arithmetic expression, e.g. '2026 - 2019'.",
     "parameters": {"type": "object",
                    "properties": {"expression": {"type": "string"}},
                    "required": ["expression"], "additionalProperties": False}},
]

In [4]:
goal = "How old is Scaler as a company in 2026? Search first, then compute."
messages = [{"role": "user", "content": goal}]

resp = client.responses.create(model=MODEL, input=messages, tools=tools)

# The model did NOT answer — it chose to call a tool:
for item in resp.output:
    if item.type == "function_call":
        print("agent decided to call:", item.name, "with", item.arguments)
    else:
        print(item.type)

reasoning
agent decided to call: web_search with {"query":"Scaler company founded year Scaler Academy founded 2019"}


In [5]:
def run_agent(goal: str, max_steps: int = 6) -> str:
    """One agent = one LLM looping (think -> act -> observe) until it stops calling tools."""
    messages = [{"role": "user", "content": goal}]

    for step in range(max_steps):
        resp = client.responses.create(model=MODEL, input=messages, tools=tools)
        messages += resp.output                      # remember the model's turn

        calls = [o for o in resp.output if o.type == "function_call"]
        if not calls:                                # no tool call -> the agent is done
            return resp.output_text

        for call in calls:                           # the AGENT picked these tools itself
            args = json.loads(call.arguments)
            result = TOOL_IMPL[call.name](**args)
            print(f"  [step {step}] {call.name}({args}) -> {result}")
            messages.append({"type": "function_call_output",
                             "call_id": call.call_id, "output": result})

    return "Stopped: hit max steps."

answer = run_agent("How old is Scaler as a company in 2026? Search first, then compute.")
print("\nFINAL ANSWER:\n", answer)


  [step 0] web_search({'query': 'Scaler founded year Scaler company founded year Scaler Academy'}) -> Top result for 'Scaler founded year Scaler company founded year Scaler Academy': Scaler is an edtech platform founded in 2019.
  [step 1] web_search({'query': 'Scaler founded 2019 edtech platform founded year'}) -> Top result for 'Scaler founded 2019 edtech platform founded year': Scaler is an edtech platform founded in 2019.

FINAL ANSWER:
 - Search result: Scaler was founded in 2019.
- Calculation: 2026 - 2019 = 7.
- Conclusion: Scaler is about 7 years old in 2026.


# Part 2

In [6]:
def ask(role: str, content: str) -> str:
    """One specialized agent = one LLM call with its own job description (`instructions`)."""
    return client.responses.create(model=MODEL, instructions=role, input=content).output_text

In [7]:
def agentic_system(task: str) -> str:
    # 1. PLAN — decompose
    subtasks = [l.strip() for l in ask(
        "You are a Planner. Break the task into 2-3 concrete subtasks, one per line, no numbering.",
        task).splitlines() if l.strip()]
    pretty_print("Plan:", subtasks)

    # 2. RESEARCH — delegate each subtask
    notes = ""
    for st in subtasks:
        notes += f"\n## {st}\n{ask('You are a Researcher. Give 2-3 crisp factual bullets.', st)}\n"

    # 3. WRITE — synthesize
    draft = ask("You are a Writer. Using the notes, write a tight ~150-word answer.",
                f"Task: {task}\n\nNotes:\n{notes}")

    # 4. CRITIQUE + 5. REVISE — self-evaluate and improve
    review = ask("You are an Editor. Reply 'APPROVED' if good, else list fixes.", draft)
    if review.strip().upper() != "APPROVED":
        draft = ask("You are a Writer. Revise with the feedback, ~150 words.",
                    f"Notes:\n{notes}\n\nDraft:\n{draft}\n\nFeedback:\n{review}")
    return draft

pretty_print(agentic_system("Explain why vector databases matter for RAG."))

Plan: ['- Identify why retrieval quality matters in RAG: how embeddings capture
semantic meaning, affect the relevance of retrieved passages, and influence the
faithfulness and accuracy of the generated answers.', '- Explain what a vector
database is and how it differs from traditional databases: it stores high-
dimensional embeddings and uses approximate nearest neighbor indexing for fast
semantic retrieval, with different performance/consistency trade-offs.', '-
Describe how vector databases enable practical RAG workflows: semantic search
over large corpora, ranking and selecting context by similarity, handling
document chunks and context windows, and maintaining data with updates and
embedding drift (including trade-offs like recall vs latency and embedding
maintenance).']
Vector databases matter for retrieval-augmented generation (RAG) because
retrieval quality shapes answer usefulness. Semantic encoding captures synonyms,
paraphrases, and contextual cues. As a result, query–doc si